# 5. Connect evidence, constraints, and model selection

**Prerequisite:** Lessons 2–4. **You will learn:** hard gates, ranking, infeasibility, evidence traces, and a reproducible application handoff. Allow 30–45 minutes. This capstone calls the actual repository selection functions on saved measurements; it does not retrain candidates or claim a new hardware benchmark.

**Execution:** Run from the repository or this folder, in order, using the installed project environment. The default lesson needs no downloads, credentials, GPU, or existing model checkpoint. Toy examples are teaching demonstrations, not the real-data study.

**Learning loop:** Read the theory, predict the output, run the example, attempt the exercise, then open the worked answer.


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'reports/results.json').is_file())
print('Repository:', ROOT.name)


Repository: Vision


## 1. Turn a deployment request into a contract
“Make it small and accurate” is ambiguous. A useful request states minimum validation quality, minimum quality during a sensor outage, maximum storage, and maximum p95 duration. A hard gate rejects candidates outside any bound. Ranking only compares candidates that pass every gate.

The pilot ranks passing candidates using 70% clean validation F1 and 30% outage F1, breaking ties with lower latency. This is a project policy, not a universal best objective. Set the policy before reading test scores.


In [2]:
from edge_agent_lab.pilot import Objective, feasible, rank_candidates
results = json.loads((ROOT / 'reports/results.json').read_text())
search = results['search']
objective = Objective(**search['objective'])
print('Saved deployment contract:', objective.model_dump())
rows = search['candidates']
for row in rows:
    print(row['candidate'], '| feasible:', feasible(row, objective))
assert all(feasible(row, objective) == row['feasible'] for row in rows)


Saved deployment contract: {'min_macro_f1': 0.9, 'max_p95_ms': 2.0, 'max_bytes': 32000, 'min_dropout_f1': 0.75, 'max_trials': 3, 'max_seconds': 900.0}
width-6-robust | feasible: True
width-6-base | feasible: True
width-4-robust | feasible: True


## 2. Explain rejection with margins
A quality margin is measured F1 minus the minimum; a resource margin is the maximum minus the measurement. Nonnegative margins pass. Exposing the margins makes a rejection reviewable instead of merely printing “failed.”

The inputs below are saved measurements. Recomputing margins checks decision logic without rerunning data collection.


In [3]:
def margins(row, target):
    return {
        'quality': row['validation']['macro_f1'] - target.min_macro_f1,
        'outage_quality': row['dropout_macro_f1'] - target.min_dropout_f1,
        'latency_ms': target.max_p95_ms - row['benchmark']['p95_ms'],
        'storage_bytes': target.max_bytes - row['benchmark']['state_dict_bytes'],
    }
for row in rows:
    print(row['candidate'], {k: round(v, 4) for k, v in margins(row, objective).items()})
ranked = rank_candidates(rows, objective)
winner = ranked[0] if ranked else None
print('Recomputed ranking winner:', winner['candidate'] if winner else None)
# This applies the pilot ranking function to the saved search candidate set.
# It is a teaching recomputation, not a new on-demand search run.


width-6-robust {'quality': 0.0589, 'outage_quality': 0.1711, 'latency_ms': 1.9132, 'storage_bytes': 10598}
width-6-base {'quality': 0.0576, 'outage_quality': 0.182, 'latency_ms': 1.9129, 'storage_bytes': 10598}
width-4-robust {'quality': 0.0577, 'outage_quality': 0.1494, 'latency_ms': 1.94, 'storage_bytes': 18214}
Recomputed ranking winner: width-6-base


## 3. No feasible candidate is a complete answer
An empty feasible set should stop promotion. Lowering a requirement silently would violate the deployment request. Here we deliberately tighten storage to one byte, retaining the other requirements.

`model_dump()` makes a plain dictionary from the validated objective. Reconstructing `Objective` validates the changed value as well.


In [4]:
tight = Objective(**{**objective.model_dump(), 'max_bytes': 1})
rejected = rank_candidates(rows, tight)
print('One-byte contract candidates:', len(rejected))
assert rejected == []


One-byte contract candidates: 0


## 4. Make the decision reviewable
A useful decision artifact records its evidence source, objective, selection split, accepted candidates, and explicit status. The artifact below stays in memory so the lesson does not overwrite original reports.

The real workflow adds dataset and checkpoint hashes, environment details, durable checkpoints, and experiment budgets. A full deployment also needs target-device measurement and application integration.


In [5]:
import hashlib
evidence = ROOT / 'reports/results.json'
decision = {
    'kind': 'tutorial_recomputation_from_saved_measurements',
    'evidence_sha256': hashlib.sha256(evidence.read_bytes()).hexdigest(),
    'selection_split': 'validation',
    'test_used_for_selection': False,
    'objective': objective.model_dump(),
    'status': 'selected' if winner else 'no_feasible_candidate',
    'winner': winner['candidate'] if winner else None,
    'accepted': [row['candidate'] for row in ranked],
}
print(json.dumps(decision, indent=2))


{
  "kind": "tutorial_recomputation_from_saved_measurements",
  "evidence_sha256": "b382e4a91a7e8103fa6ab03709309848592cd80b8a4ee3c777e6e82ce588636d",
  "selection_split": "validation",
  "test_used_for_selection": false,
  "objective": {
    "min_macro_f1": 0.9,
    "max_p95_ms": 2.0,
    "max_bytes": 32000,
    "min_dropout_f1": 0.75,
    "max_trials": 3,
    "max_seconds": 900.0
  },
  "status": "selected",
  "winner": "width-6-base",
  "accepted": [
    "width-6-base",
    "width-6-robust",
    "width-4-robust"
  ]
}


## Exercise: change a requirement, not the evidence
Increase the outage-quality floor to 0.95 by constructing a new validated objective. Recompute the feasible set. Which margins reject each candidate? Does a higher test score justify bypassing the floor?

<details><summary>Worked answer</summary>

Use `stricter = Objective(**{**objective.model_dump(), 'min_dropout_f1': .95})`, then call `rank_candidates(rows, stricter)` and inspect `margins(row, stricter)`. The result depends on the saved measurements; inspect it rather than assuming a winner. Test scores cannot bypass a predeclared gate without turning test reporting into model selection.
</details>

## 5. Move from the lesson to the real application
The complete local workflow is:

1. Prepare the real dataset and validate provenance and subject separation.
2. Train bounded candidates and preserve checkpoints.
3. Investigate sensor incidents through retrieval and typed tools.
4. Measure validation quality, outage quality, storage, and latency.
5. Select only feasible candidates, then report held-out test results.
6. Export, verify prediction parity, and repeat timing on target hardware.

From the repository root, the existing project commands are:

```bash
edge-lab prepare
edge-lab train --seeds 7 --epochs 40 --compact
edge-lab steward-eval
edge-lab pilot
edge-lab evaluate
edge-lab report
```

These commands are **not run by this tutorial**. Preparation downloads the real dataset, training takes longer than these examples, and evaluation/reporting writes local artifacts. Use [the reproduction guide](../../docs/REPRODUCING.md) for full prerequisites and output locations.

**Final challenge:** Write a short deployment proposal stating your target device, four gates, measurement scope, incident checks, and what should happen if no candidate passes. Keep battery, peak-memory, and target-device claims out of the proposal until measured.

**Application bridge:** Compare this lesson with [Edge Pilot](../03_edge_pilot.ipynb), `pilot.py`, and `search.py`. The controller is a bounded engineering policy; the optional language-model system is a separate decision component. You now have a route from sensor arrays through training, optimization, investigation, and a reviewable deployment decision.
